In [1]:
import pandas as pd
import matplotlib.pyplot as plt

# Paths to your files
file1 = r"C:/datasources/corpus_file_list_by_size.txt"
file2 = r"C:/datasources/blackvault_corpus_file_list_by_size.txt"

# Load as dataframes
df1 = pd.read_csv(file1, sep=" ", names=["size", "filename"], engine="python")
df2 = pd.read_csv(file2, sep=" ", names=["size", "filename"], engine="python")

# Sort by size (largest first)
df1_sorted = df1.sort_values(by="size", ascending=False)
df2_sorted = df2.sort_values(by="size", ascending=False)

# Remove smallest files

df1 = df1[df1["size"] >= 650].copy()
df2 = df2[df2["size"] >= 650].copy()

# Define substrings to remove
heavy_patterns = ["s3.documentcloud", "files.usrtk", "wp-content_uploads"]

# Build regex pattern (joined with | means "OR")
pattern = "|".join(heavy_patterns)

# Filter both dataframes
df1 = df1[~df1["filename"].str.contains(pattern, na=False)].copy()
df2 = df2[~df2["filename"].str.contains(pattern, na=False)].copy()

# Cut pdfs from black vault
df2 = df2[~df2["filename"].str.contains("pdf", na=False)].copy()

total_size_df1 = df1["size"].sum()
total_size_df2 = df2["size"].sum()

print("Total size of df1:", total_size_df1)
print("Total size of df2:", total_size_df2)

Total size of df1: 1913759473.0
Total size of df2: 145435276.0


In [9]:
# Sort df1 and save
df1_sorted = df1.sort_values(by="size", ascending=False)
df1_sorted.to_csv("C:/datasources/corpus_main_list.txt", sep=" ", index=False, header=False)

# Sort df2 and save
df2_sorted = df2.sort_values(by="size", ascending=False)
df2_sorted.to_csv("C:/datasources/corpus_blackvault_list.txt", sep=" ", index=False, header=False)

In [10]:
dfscrub = pd.read_csv("C:/datasources/corpus_main_list.txt", sep=" ", names=["size", "filename"], engine="python")
total_size_scrub = dfscrub["size"].sum()

print("Total size of df1:", total_size_scrub)

Total size of df1: 1575190731.0


In [2]:
# Jupyter cell: corpus cleaner
# 1) If needed, install deps right in the notebook:
# %pip install beautifulsoup4 lxml ftfy chardet tqdm

import os, re, csv, shutil, html, unicodedata
from pathlib import Path
from bs4 import BeautifulSoup
from ftfy import fix_text
import chardet
from tqdm import tqdm

# --- CONFIG ---
INPUT_DIR  = Path("C:/datasources/ai_corpus_slimmer/corpus_beta_set_slim")   # <- change me
OUTPUT_DIR = Path("C:/datasources/ai_corpus_slimmer/ai_corpus_slimmer_clean")  # <- change me
REJECT_DIR = OUTPUT_DIR / "rejected"
LOG_CSV    = OUTPUT_DIR / "clean_log.csv"
EXTS       = {".txt", ".html", ".htm"}
WORDS_IN_A_ROW_THRESHOLD = 60
ALPHA_TOKEN_MIN_FRACTION = 0.80   # in a 100-word window, ≥80% tokens should be alphabetic
MAX_NONASCII_FRACTION    = 0.20   # in a 100-word window, ≤20% chars non-ASCII
DRY_RUN = False  # True = do not write/move; just report

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REJECT_DIR.mkdir(parents=True, exist_ok=True)

def read_text_with_detection(p: Path) -> str:
    raw = p.read_bytes()
    guess = chardet.detect(raw) or {}
    enc = guess.get("encoding") or "utf-8"
    try:
        return raw.decode(enc, errors="replace")
    except LookupError:
        return raw.decode("utf-8", errors="replace")

def strip_html(text: str, ext: str) -> str:
    # Heuristic: if looks like HTML or extension is html/htm -> parse
    looks_like_html = ("<html" in text[:1000].lower()) or ("</p>" in text.lower()) or ("<body" in text.lower())
    if ext in {".html", ".htm"} or looks_like_html:
        soup = BeautifulSoup(text, "lxml")  # fallbacks to html.parser if lxml missing
        for tag in soup(["script", "style", "noscript"]):
            tag.decompose()
        text = soup.get_text(separator=" ")
    # Unescape entities
    return html.unescape(text)

def normalize_and_clean(text: str) -> str:
    # Fix mojibake and weird unicode
    text = fix_text(text)
    # Normalize to NFC
    text = unicodedata.normalize("NFC", text)
    # Remove control / non-printing (keep basic whitespace)
    text = "".join(ch if (ch.isprintable() or ch in "\n\t ") else " " for ch in text)
    # Replace intraword punctuation like "securIty^^^>^^" with spaces around runs
    text = re.sub(r"[^\w\s]", " ", text)
    # Collapse runs of underscores/dashes/etc. to a space
    text = re.sub(r"[_\-+=~^`|\\/<>{}\[\]()*#%$@:;.,!?]{2,}", " ", text)
    # Collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()
    return text

_word_re = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?")  # alpha tokens; allow simple apostrophes

def has_natural_language_run(clean_text: str,
                             window_size: int = WORDS_IN_A_ROW_THRESHOLD,
                             alpha_min_frac: float = ALPHA_TOKEN_MIN_FRACTION,
                             max_nonascii_frac: float = MAX_NONASCII_FRACTION) -> bool:
    # Tokenize to words
    tokens = _word_re.findall(clean_text)
    if len(tokens) < window_size:
        return False

    # Precompute for sliding window: mark alpha-only tokens
    is_alpha = [t.isalpha() for t in tokens]

    # For non-ASCII fraction per window, work on character slices:
    # Make a quick index of cumulative counts to avoid O(n^2).
    chars = clean_text
    # To approximate per-window non-ASCII, we map token indices to char spans.
    # Simpler heuristic: compute on token strings themselves.
    token_nonascii_frac = [sum(ord(c) > 127 for c in t)/max(1,len(t)) for t in tokens]

    # Slide 100-word window
    alpha_count = sum(is_alpha[:window_size])
    nonascii_avg = sum(token_nonascii_frac[:window_size]) / window_size

    if alpha_count / window_size >= alpha_min_frac and nonascii_avg <= max_nonascii_frac:
        return True

    for i in range(window_size, len(tokens)):
        # remove left, add right
        alpha_count += is_alpha[i] - is_alpha[i - window_size]
        nonascii_avg += (token_nonascii_frac[i] - token_nonascii_frac[i - window_size]) / window_size
        if alpha_count / window_size >= alpha_min_frac and nonascii_avg <= max_nonascii_frac:
            return True

    return False

def is_garbled(clean_text: str) -> bool:
    # Reject if too short overall
    if len(clean_text) < 400:  # ~a few sentences
        return True
    # Ratio of letters to all non-space chars; if extremely low, it's junk
    nospace = clean_text.replace(" ", "")
    if not nospace:
        return True
    alpha = sum(c.isalpha() for c in nospace)
    if alpha / len(nospace) < 0.55:
        return True
    return False

def relative_output_path(src: Path) -> Path:
    rel = src.relative_to(INPUT_DIR)
    return OUTPUT_DIR / rel.with_suffix(".txt")  # save everything as clean .txt

files = [p for p in INPUT_DIR.rglob("*") if p.is_file() and p.suffix.lower() in EXTS]

stats = {
    "total_files": len(files),
    "clean_kept": 0,
    "rejected": 0,
}

log_rows = []
pbar = tqdm(files, desc="Cleaning files")
for src in pbar:
    try:
        raw = read_text_with_detection(src)
        stripped = strip_html(raw, src.suffix.lower())
        cleaned = normalize_and_clean(stripped)

        ok_run = has_natural_language_run(cleaned)
        junky  = is_garbled(cleaned)

        decision = "keep" if (ok_run and not junky) else "reject"

        if decision == "keep":
            dst = relative_output_path(src)
            if not DRY_RUN:
                dst.parent.mkdir(parents=True, exist_ok=True)
                with open(dst, "w", encoding="utf-8", newline="\n") as f:
                    f.write(cleaned)
            stats["clean_kept"] += 1
        else:
            # Move original to rejected/ (keep relative path)
            rej = REJECT_DIR / src.relative_to(INPUT_DIR)
            if not DRY_RUN:
                rej.parent.mkdir(parents=True, exist_ok=True)
                shutil.move(str(src), str(rej))
            stats["rejected"] += 1

        log_rows.append({
            "source_path": str(src),
            "decision": decision,
            "clean_chars": len(cleaned),
            "total_words": len(_word_re.findall(cleaned)),
        })

    except Exception as e:
        log_rows.append({
            "source_path": str(src),
            "decision": f"error: {type(e).__name__}: {e}",
            "clean_chars": 0,
            "total_words": 0,
        })

# Write log
if not DRY_RUN:
    LOG_CSV.parent.mkdir(parents=True, exist_ok=True)
    with open(LOG_CSV, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["source_path","decision","clean_chars","total_words"])
        w.writeheader()
        w.writerows(log_rows)

print("DONE\n", stats)
print(f"Log at: {LOG_CSV}")
print(f"Rejected originals moved under: {REJECT_DIR}")
print(f"Cleaned texts under: {OUTPUT_DIR}")



Cleaning files: 100%|████████████████████████████████████████████████████████████| 12215/12215 [07:09<00:00, 28.42it/s]

DONE
 {'total_files': 12215, 'clean_kept': 12199, 'rejected': 14}
Log at: C:\datasources\ai_corpus_slimmer\ai_corpus_slimmer_clean\clean_log.csv
Rejected originals moved under: C:\datasources\ai_corpus_slimmer\ai_corpus_slimmer_clean\rejected
Cleaned texts under: C:\datasources\ai_corpus_slimmer\ai_corpus_slimmer_clean
